In [2]:
# ==========================================
# CELDA 1: Setup
# ==========================================
import sys, os

!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git

PROJECT_ROOT = '/content/proyecto-transformacion-texto-imagen'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

!pip install -q transformers torch scikit-learn polars joblib optuna

print("✓ Entorno configurado")


Cloning into 'proyecto-transformacion-texto-imagen'...
remote: Enumerating objects: 300, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 300 (delta 1), reused 0 (delta 0), pack-reused 296 (from 1)
Receiving objects: 100% (300/300), 6.10 MiB | 20.34 MiB/s, done.
Resolving deltas: 100% (151/151), done.
✓ Entorno configurado


In [3]:
# ==========================================
# CELDA 2: Carga y generación de corpus filtrado
# ==========================================
import polars as pl
import numpy as np
from features.pandemic_stopwords import apply_pandemic_filter, count_pandemic_terms

# Cargar datos originales
train = pl.read_csv(f'{PROJECT_ROOT}/data/processed/train.csv')
test  = pl.read_csv(f'{PROJECT_ROOT}/data/processed/test.csv')

X_train_orig = train.get_column('text').to_list()
y_train      = train.get_column('manual_classification').to_numpy()
X_test_orig  = test.get_column('text').to_list()
y_test       = test.get_column('manual_classification').to_numpy()

# Aplicar filtro de pandemia
X_train_nopand = apply_pandemic_filter(X_train_orig)
X_test_nopand  = apply_pandemic_filter(X_test_orig)

# ── Estadística del filtro ──────────────────────────────────
n_afectados_train = sum(1 for t in X_train_orig if count_pandemic_terms(t) > 0)
n_afectados_test  = sum(1 for t in X_test_orig  if count_pandemic_terms(t) > 0)

print(f"Textos con términos de pandemia:")
print(f"  Train: {n_afectados_train}/{len(X_train_orig)} ({n_afectados_train/len(X_train_orig)*100:.1f}%)")
print(f"  Test:  {n_afectados_test}/{len(X_test_orig)}  ({n_afectados_test/len(X_test_orig)*100:.1f}%)")

# Guardar corpus filtrado para reutilización futura
train_nopand = train.with_columns(pl.Series('text', X_train_nopand))
test_nopand  = test.with_columns(pl.Series('text', X_test_nopand))

train_nopand.write_csv(f'{PROJECT_ROOT}/data/processed/train_nopandemic.csv')
test_nopand.write_csv(f'{PROJECT_ROOT}/data/processed/test_nopandemic.csv')
print("\n✓ Corpus filtrado guardado en data/processed/")

# Ejemplo de transformación
print("\n── Ejemplo de limpieza ──")
idx = next(i for i, t in enumerate(X_train_orig) if count_pandemic_terms(t) > 0)
print(f"Original:  {X_train_orig[idx][:120]}")
print(f"Filtrado:  {X_train_nopand[idx][:120]}")


Textos con términos de pandemia:
  Train: 223/908 (24.6%)
  Test:  23/114  (20.2%)

✓ Corpus filtrado guardado en data/processed/

── Ejemplo de limpieza ──
Original:  Buenos días, me siento muy bien conmigo misma, bien con mis pensamientos, mi familia, mi entorno y mis amigos. Pero ya n
Filtrado:  Buenos días, me siento muy bien conmigo misma, bien con mis pensamientos, mi familia, mi entorno y mis amigos. Pero ya n


In [5]:
import os
import json
import torch
import numpy as np
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

# ==============================
# Reproducibilidad
# ==============================
set_seed(42)


# ==============================
# Dataset
# ==============================
class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


# ==============================
# Métricas
# ==============================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }


# ==============================
# Fine-Tuning Principal
# ==============================
def run_finetuning(train_texts, train_labels, val_texts, val_labels, model_name):

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    train_ds = DepressionDataset(train_texts, train_labels, tokenizer)
    val_ds = DepressionDataset(val_texts, val_labels, tokenizer)

    safe_model_name = model_name.replace("/", "_")
    output_dir = f"models/checkpoints/{safe_model_name}"

    training_args = TrainingArguments(
        output_dir=output_dir,

        # Estrategia de evaluación
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        # Optimización
        learning_rate=2.3048016342698774e-05,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.06379717552110609,
        warmup_ratio=0.0013564417879888579,

        # Selección del mejor modelo
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,

        # Control de checkpoints
        save_total_limit=1,
        fp16=True,

        # Reproducibilidad
        seed=42,
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    # ==========================
    # Evaluación Final
    # ==========================
    final_metrics = trainer.evaluate()
    print("\nFinal Evaluation Metrics:")
    print(final_metrics)

    # Guardar métricas en JSON
    os.makedirs("results", exist_ok=True)
    with open(f"results/{safe_model_name}_metrics.json", "w") as f:
        json.dump(final_metrics, f, indent=4)

    # ==========================
    # Matriz de Confusión
    # ==========================
    predictions = trainer.predict(val_ds)
    preds = np.argmax(predictions.predictions, axis=1)
    cm = confusion_matrix(val_labels, preds)

    print("\nConfusion Matrix:")
    print(cm)

    return trainer

In [7]:
# ==========================================
# CELDA 3: Fine-tuning BETO sin términos de pandemia
# (Mismos hiperparámetros óptimos del Exp 4.1)
# ==========================================

print("="*55)
print("EXP 4.2: BETO Fine-Tuning SIN términos de pandemia")
print("="*55)

trainer_nopand = run_finetuning(
    X_train_nopand, y_train,
    X_test_nopand,  y_test,
)

print("\n✅ Exp 4.2 completado")


EXP 4.2: BETO Fine-Tuning SIN términos de pandemia


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.636303,0.538272,0.754386,0.753049,0.715584,0.723684
2,0.464610,0.491015,0.736842,0.722222,0.718182,0.719948
3,0.293615,0.524615,0.789474,0.777813,0.782143,0.779710
4,0.139150,0.702707,0.771930,0.788621,0.725649,0.736111
5,0.059650,0.732476,0.736842,0.723684,0.709740,0.714333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Final Evaluation Metrics:
{'eval_loss': 0.5250725746154785, 'eval_accuracy': 0.7894736842105263, 'eval_precision_macro': 0.7778132992327366, 'eval_recall_macro': 0.7821428571428571, 'eval_f1_macro': 0.7797101449275362, 'eval_runtime': 0.4248, 'eval_samples_per_second': 268.34, 'eval_steps_per_second': 18.831, 'epoch': 5.0}

Confusion Matrix:
[[57 13]
 [11 33]]

✅ Exp 4.2 completado


In [13]:
# ==========================================
# CELDA 4 CORREGIDA: Comparación con consistencia de datos
# ==========================================
import joblib, json, os
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

def extraer_embeddings(texts, ckpt_path, batch_size=16):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
    model     = AutoModel.from_pretrained(ckpt_path).to(device)
    model.eval()
    all_emb = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"  Embeddings"):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model(**inputs)
            all_emb.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_emb)

# ── PASO 1: Identificar el mejor checkpoint del Exp 4.2 ────
ckpt_base = f'{PROJECT_ROOT}/models/checkpoints'

# El mejor checkpoint del Exp 4.2 es época 3 → step 171 (57 steps/época × 3)
# Ajusta este path según lo que imprima arriba
CKPT_42 = f'{ckpt_base}/dccuchile_bert-base-spanish-wwm-cased/checkpoint-171'

# Si no existe checkpoint-171, usa la raíz del directorio
if not os.path.exists(CKPT_42):
    CKPT_42 = f'{ckpt_base}/dccuchile-bert-base-spanish-wwm-cased'
    print(f"\n⚠️  checkpoint-171 no encontrado, usando: {CKPT_42}")
else:
    print(f"\n✓ Usando mejor checkpoint (época 3): {CKPT_42}")

# ── PASO 2: Exp 4.1 desde PKL (textos ORIGINALES) ──────────
print("\n[Exp 4.1] Cargando embeddings desde PKL...")
pkg_41         = joblib.load(f'{PROJECT_ROOT}/models/best_model/Exp_4.1_BETO_HPO_embeddings.pkl')
X_train_emb_41 = pkg_41['X_train']
X_test_emb_41  = pkg_41['X_test']
y_train        = pkg_41['y_train']
y_test         = pkg_41['y_test']
print(f"✓ Shape → train: {X_train_emb_41.shape} | test: {X_test_emb_41.shape}")

# ── PASO 3: Exp 4.2 desde checkpoint (textos SIN pandemia) ─
print(f"\n[Exp 4.2] Extrayendo embeddings desde checkpoint...")
print(f"  IMPORTANTE: Se usan X_train_nopand y X_test_nopand")

# Verificar que X_train_nopand existe en memoria
# Si reiniciaste el kernel, vuelve a generarlos:
try:
    _ = X_train_nopand
except NameError:
    from features.pandemic_stopwords import apply_pandemic_filter
    import polars as pl
    train_df = pl.read_csv(f'{PROJECT_ROOT}/data/processed/train_nopandemic.csv')
    test_df  = pl.read_csv(f'{PROJECT_ROOT}/data/processed/test_nopandemic.csv')
    X_train_nopand = train_df.get_column('text').to_list()
    X_test_nopand  = test_df.get_column('text').to_list()
    print("  (Corpus filtrado recargado desde CSV)")

X_train_emb_42 = extraer_embeddings(X_train_nopand, CKPT_42)
X_test_emb_42  = extraer_embeddings(X_test_nopand,  CKPT_42)
print(f"✓ Shape → train: {X_train_emb_42.shape} | test: {X_test_emb_42.shape}")

# ── PASO 4: Guardar PKL del Exp 4.2 ────────────────────────
pkg_42 = {
    'X_train': X_train_emb_42, 'y_train': y_train,
    'X_test':  X_test_emb_42,  'y_test':  y_test,
    'checkpoint': CKPT_42,
    'nota': 'Embeddings generados con textos sin términos de pandemia'
}
path_42 = f'{PROJECT_ROOT}/models/best_model/Exp_4.2_BETO_HPO_SinPandemia.pkl'
joblib.dump(pkg_42, path_42)
print(f"\n✓ PKL Exp 4.2 guardado en: {path_42}")

# ── PASO 5: Comparación con LogisticRegression ─────────────
# REGLA: Cada LogReg se entrena y evalúa con embeddings del MISMO experimento
print("\n" + "="*55)
print("📊 COMPARACIÓN FINAL - Sesgo de Pandemia")
print("="*55)
print(f"{'Experimento':<30} {'F1-Macro':>10} {'vs Baseline':>13}")
print("-"*55)
print(f"{'Baseline (sin HPO)':<30} {'0.7411':>10}")

resultados = {}
configs = [
    ('4.1 CON pandemia', X_train_emb_41, X_test_emb_41, y_train, y_test),
    ('4.2 SIN pandemia', X_train_emb_42, X_test_emb_42, y_train, y_test),
]

for nombre, X_tr, X_te, y_tr, y_te in configs:
    clf = LogisticRegression(class_weight='balanced',
                             solver='liblinear', random_state=42)
    clf.fit(X_tr, y_tr)           # ← Entrenado con embeddings del mismo exp
    preds = clf.predict(X_te)     # ← Evaluado con embeddings del mismo exp
    f1    = f1_score(y_te, preds, average='macro')
    resultados[nombre] = {'f1_macro': round(f1, 4), 'preds': preds}
    delta = f1 - 0.7411
    print(f"{'Exp ' + nombre:<30} {f1:>10.4f}  ({delta:+.4f})")

# ── PASO 6: Reporte detallado por clase ────────────────────
print("\n── Exp 4.1 CON pandemia ──")
print(classification_report(y_test, resultados['4.1 CON pandemia']['preds'],
                             target_names=['No Depresivo', 'Depresivo']))

print("\n── Exp 4.2 SIN pandemia ──")
print(classification_report(y_test, resultados['4.2 SIN pandemia']['preds'],
                             target_names=['No Depresivo', 'Depresivo']))

# ── PASO 7: Guardar resultados ─────────────────────────────
comp = {
    'baseline':              0.7411,
    'exp_4.1_con_pandemia':  resultados['4.1 CON pandemia']['f1_macro'],
    'exp_4.2_sin_pandemia':  resultados['4.2 SIN pandemia']['f1_macro'],
    'delta_4.1_vs_baseline': round(resultados['4.1 CON pandemia']['f1_macro'] - 0.7411, 4),
    'delta_4.2_vs_baseline': round(resultados['4.2 SIN pandemia']['f1_macro'] - 0.7411, 4),
    'delta_4.1_vs_4.2':      round(resultados['4.1 CON pandemia']['f1_macro'] -
                                   resultados['4.2 SIN pandemia']['f1_macro'], 4),
    'checkpoint_42_usado':   CKPT_42,
}
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)
with open(f'{PROJECT_ROOT}/results/comparacion_sesgo_pandemia.json', 'w') as f:
    json.dump(comp, f, indent=4)

print(f"\n✓ Resultados guardados en results/comparacion_sesgo_pandemia.json")



✓ Usando mejor checkpoint (época 3): /content/proyecto-transformacion-texto-imagen/models/checkpoints/dccuchile_bert-base-spanish-wwm-cased/checkpoint-171

[Exp 4.1] Cargando embeddings desde PKL...
✓ Shape → train: (908, 768) | test: (114, 768)

[Exp 4.2] Extrayendo embeddings desde checkpoint...
  IMPORTANTE: Se usan X_train_nopand y X_test_nopand


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /content/proyecto-transformacion-texto-imagen/models/checkpoints/dccuchile_bert-base-spanish-wwm-cased/checkpoint-171
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Embeddings: 100%|██████████| 57/57 [00:06<00:00,  8.62it/s]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /content/proyecto-transformacion-texto-imagen/models/checkpoints/dccuchile_bert-base-spanish-wwm-cased/checkpoint-171
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Embeddings: 100%|██████████| 8/8 [00:00<00:00,  9.48it/s]


✓ Shape → train: (908, 768) | test: (114, 768)

✓ PKL Exp 4.2 guardado en: /content/proyecto-transformacion-texto-imagen/models/best_model/Exp_4.2_BETO_HPO_SinPandemia.pkl

📊 COMPARACIÓN FINAL - Sesgo de Pandemia
Experimento                      F1-Macro   vs Baseline
-------------------------------------------------------
Baseline (sin HPO)                 0.7411
Exp 4.1 CON pandemia               0.7881  (+0.0470)
Exp 4.2 SIN pandemia               0.5117  (-0.2294)

── Exp 4.1 CON pandemia ──
              precision    recall  f1-score   support

No Depresivo       0.84      0.83      0.83        70
   Depresivo       0.73      0.75      0.74        44

    accuracy                           0.80       114
   macro avg       0.79      0.79      0.79       114
weighted avg       0.80      0.80      0.80       114


── Exp 4.2 SIN pandemia ──
              precision    recall  f1-score   support

No Depresivo       0.62      0.69      0.65        70
   Depresivo       0.41      0.34  